In [68]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib
import tldextract

In [69]:
df = pd.read_csv('csvs/urldata.csv')
df.drop(columns=['Unnamed: 0'],inplace=True)

In [70]:
print(df.columns)

Index(['url', 'label', 'result'], dtype='object')


In [71]:
df.head()

,url,label,result
0,https://www.google.com,benign,0
1,https://www.youtube.com,benign,0
2,https://www.facebook.com,benign,0
3,https://www.baidu.com,benign,0
4,https://www.wikipedia.org,benign,0


In [72]:
risk_words = ['login','admin','update','secure','account']

In [73]:
def get_features(url):
    extracted_object = tldextract.extract(url)
    size = len(url)
    https = url.startswith('https://')
    subdomain = extracted_object.subdomain
    #domain = extracted_object.domain
    suffix = extracted_object.suffix
    dots = url.count('.')
    dashes = url.count('-')
    numbers = sum(c.isdigit() for c in url)
    risky=0
    for d in risk_words:
        if d in url:
            risky+=1
    f_slash = sum(c == '/' for c in url)
    b_slash = sum(c == '\\' for c in url)
    underscore = sum(c == '_' for c in url)
    at = True if '@' in url else False
    #return size, https, subdomain, suffix, dots, dashes, numbers, login, f_slash, b_slash, underscore, at
    return {
        'size' : size,
        'https' : https,
        'subdomain' : subdomain,
        'suffix' : suffix,
        'dots' : dots,
        'dashes' : dashes,
        'numbers' : numbers,
        'risky' : risky,
        'f_slash' : f_slash,
        'b_slash' : b_slash,
        'underscore' : underscore,
        'at' : at
    }
    

In [74]:
'''
final_df = []
for u in df['url']:
    data = get_features(u)
final_df = pd.DataFrame(final_df)
'''



"\nfinal_df = []\nfor u in df['url']:\n    data = get_features(u)\nfinal_df = pd.DataFrame(final_df)\n"

In [75]:
df_features = df['url'].apply(get_features).apply(pd.Series)

In [76]:
final_df = pd.concat([df,df_features], axis=1)

In [77]:
final_df.head()

,url,label,result,size,https,subdomain,suffix,dots,dashes,numbers,risky,f_slash,b_slash,underscore,at
0,https://www.google.com,benign,0,22,True,www,com,2,0,0,0,2,0,0,False
1,https://www.youtube.com,benign,0,23,True,www,com,2,0,0,0,2,0,0,False
2,https://www.facebook.com,benign,0,24,True,www,com,2,0,0,0,2,0,0,False
3,https://www.baidu.com,benign,0,21,True,www,com,2,0,0,0,2,0,0,False
4,https://www.wikipedia.org,benign,0,25,True,www,org,2,0,0,0,2,0,0,False


In [78]:
final_df.columns

Index(['url', 'label', 'result', 'size', 'https', 'subdomain', 'suffix',
       'dots', 'dashes', 'numbers', 'risky', 'f_slash', 'b_slash',
       'underscore', 'at'],
      dtype='object')

In [85]:
numerical_cols = ['https','size', 'dots', 'dashes', 'numbers', 'risky', 'f_slash', 'b_slash', 'underscore', 'at']
categorical_cols = ['subdomain', 'suffix']


In [86]:
y = final_df['result']
X = final_df[numerical_cols + categorical_cols]

In [87]:
imputer = Pipeline([('imputer', SimpleImputer())])
encoder = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), 
                     ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
ordinal = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), 
                     ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])

In [92]:
preprocessor = ColumnTransformer([('imputer', imputer, numerical_cols), 
                                  ('encoder', encoder, ['suffix']), 
                                  ('ordinal', ordinal, ['subdomain'])])

In [98]:
Xtrain,Xvalid,ytrain,yvalid = train_test_split(X,y, random_state=1, shuffle=True)

In [99]:
Xtrain = preprocessor.fit_transform(Xtrain)
Xvalid = preprocessor.transform(Xvalid)

In [100]:
model = XGBClassifier(random_state=1, n_estimators=-1,)

In [101]:
model.fit(Xtrain,ytrain, eval_set=[(Xvalid,yvalid)], verbose=False)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
predictions = model.predict(Xvalid)

: 

In [ ]:
joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(model,'model.pkl')